# 📘 Initialize POD Environment> **Applicable Environment**: Kubernetes Pod (Ubuntu base image)> **Purpose**: Complete the basic configuration of the development environment, including creating symlinks for working directories, installing and starting the SSH service, to prepare for subsequent development and debugging.## 1. Competition IntroductionThis is the AMD Radeon-hackathon-2026-07 competition project.- **Runtime Constraints**: No Docker/Podman inside the Pod, data persistence via PVC mounts, PostgreSQL managed by custom scripts- **Key Path**: Persistent directory `/workspace/persistent`, needs to be mapped to `/data` for unified service access## 2. Create Directory SymlinkSymlink the persistent volume mount point `/workspace/persistent` to `/data` to ensure all services (PostgreSQL, model files, vector database, etc.) use the unified `/data` path.```python%%bash#!/bin/bash# 1. Check if source path existsif [ ! -e /workspace/persistent ]; then    echo "❌ Error: /workspace/persistent does not exist, cannot create symlink"    exit 1fi# 2. Check /data statusif [ -L /data ]; then    echo "/data is already a symlink, pointing to $(readlink /data), skipping creation"    exit 0elif [ -d /data ] && [ ! -L /data ]; then    echo "/data is a regular directory, skipping creation"    exit 0elif [ -e /data ] && [ ! -L /data ] && [ ! -d /data ]; then    # Handle other types (such as files, sockets, etc.)    echo "/data already exists, but is neither a directory nor a symlink, skipping creation"    exit 0fi# 3. Create symlinkecho "🔗 Creating symlink /data -> /workspace/persistent"ln -s /workspace/persistent /data# 4. Verify creation resultif [ -L /data ]; then    echo "✅ Creation successful"    ls -la /dataelse    echo "❌ Creation failed, please check permissions"    exit 1fi```## 3. Install and Start SSH ServiceInstall and start OpenSSH server for remote debugging and file transfer.**Note**: If SSH is already built into the Pod, you can skip installation, but base images typically do not include it.```python%%bash#!/bin/bashset -euo pipefailSSHD_BIN="/usr/sbin/sshd"SSHD_DIR="/run/sshd"# 1. Check if sshd is installedif command -v sshd &> /dev/null || [ -x "$SSHD_BIN" ]; then    echo "✅ sshd installed: $(which sshd 2>/dev/null || echo $SSHD_BIN)"else    echo "🔧 sshd not installed, starting installation..."    sudo apt update -qq    sudo apt install -y openssh-server    echo "✅ Installation complete"fi# 2. Create runtime directory (required by some distributions)if [ ! -d "$SSHD_DIR" ]; then    sudo mkdir -p "$SSHD_DIR"    echo "📁 Creating directory $SSHD_DIR"fi# 3. Check if sshd is already runningif pgrep -x "sshd" > /dev/null; then    echo "✅ sshd service is already running"else    echo "🚀 Starting sshd..."    sudo "$SSHD_BIN"    # Verify startup    sleep 2    if pgrep -x "sshd" > /dev/null; then        echo "✅ sshd started successfully"    else        echo "❌ sshd failed to start, please check logs"        exit 1    fifi# 4. Display current statusecho "📊 SSH service status:"ps aux | grep sshd | grep -v grep || echo "⚠️ No sshd process found"```## 4. OtherOther auxiliary tools```python%%bash# treeapt -y install tree```## 5. Recommendations- **Set SSH password/key**: For remote access, configure `~/.ssh/authorized_keys` or modify `sshd_config`.- **Configure PostgreSQL environment variables**: Refer to `/data/service/pg-unires/README.md` to set `PGUSER`, `PGPASSWORD`, etc.- **Download model files**: Execute `/scripts/download_models.py` to pull Qwen quantized models to `/data/models`.---> ✅ At this point, basic system initialization is complete, and you can proceed to deploy other Uni-Resource Agent components.